# Distributed Multi-Rank Stencil & Asynchronous Halo Exchange in MPI + CUDA (Python)

4-rank 2D anisotropic reaction-diffusion solver (`mpi4py` + `numba.cuda`) on a 2 × 2 Cartesian grid with GPU halo pack/unpack, pinned host staging, an 18 × 18 shared-memory tiled stencil and dual-stream overlap.

Run on a **T4 GPU** runtime with **Runtime → Run all**.

### Cell 1 — Environment verification & mpi4py installation

In [1]:
!nvidia-smi --query-gpu=name,uuid,compute_cap,memory.total --format=csv
!mpirun --version | head -n 2
!pip install -q mpi4py numba numpy

name, uuid, compute_cap, memory.total [MiB]
Tesla T4, GPU-9f464241-9201-a2c6-af69-f3df6f6794fd, 7.5, 15360 MiB
mpirun (Open MPI) 4.1.6



### Cell 2 — Student identity configuration

In [6]:
GITHUB_USERNAME = "230103278"
assert GITHUB_USERNAME != "REPLACE_WITH_YOUR_GITHUB_USERNAME" and len(GITHUB_USERNAME.strip()) > 0, (
    "You must set GITHUB_USERNAME to your real GitHub handle."
)
print(f"Configured student GitHub identity: {GITHUB_USERNAME}")

Configured student GitHub identity: 230103278


### Cell 3 — Distributed MPI + Numba CUDA solver (`%%writefile distributed_cuda_solver.py`)

In [3]:
%%writefile distributed_cuda_solver.py
import os
import sys
import json
import math
import hashlib
import subprocess
from datetime import datetime, timezone
from types import SimpleNamespace

import numpy as np
from mpi4py import MPI
from numba import cuda, float64

# Compile-time constants for 2D shared-memory tiling (16x16 interior + 1-cell halo apron)
TILE_DIM = 16
SHARED_DIM = TILE_DIM + 2  # 18
GLOBAL_NX = 512
GLOBAL_NY = 512
STEPS = 400
LAMBDA_DAMP = 0.15

PROC_DIMS = [2, 2]
NY_LOCAL = GLOBAL_NY // PROC_DIMS[0]  # 256 owned rows per rank
NX_LOCAL = GLOBAL_NX // PROC_DIMS[1]  # 256 owned columns per rank
DX = 1.0 / (GLOBAL_NX + 1)  # 1/513: owned cells are global 1..512, Dirichlet walls sit at 0 and 513
DY = 1.0 / (GLOBAL_NY + 1)
CFL_SAFETY = 0.8  # dt = 0.8 * dx^2 / (4 * kappa_max)
REPORT_EVERY = 100
PACK_THREADS = 128

# Segment order inside the 1D halo buffers. A strip sent towards direction d carries tag d,
# so the receiver expects the tag of the opposite direction.
UP, DOWN, LEFT, RIGHT = 0, 1, 2, 3
OPPOSITE = (DOWN, UP, RIGHT, LEFT)


def _abort_on_error(exc_type, exc, tb):
    sys.__excepthook__(exc_type, exc, tb)
    MPI.COMM_WORLD.Abort(1)  # one crashed rank must not leave the other three blocked in a collective


sys.excepthook = _abort_on_error


def get_gpu_uuid() -> str:
    """Query active NVIDIA GPU UUID via nvidia-smi."""
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=uuid", "--format=csv,noheader"],
        encoding="utf-8",
    )
    return out.strip().splitlines()[0].strip()


def derive_student_seed(github_user: str, gpu_uuid: str) -> int:
    """Derive deterministic 64-bit integer seed bound to student GitHub handle and GPU UUID."""
    digest = hashlib.sha256(f"{github_user.strip()}::{gpu_uuid}".encode("utf-8")).hexdigest()
    return int(digest[:16], 16)


# ==============================================================================
# TASK 1: Distributed Cartesian Topology & Hardware Context Initialization
# ==============================================================================
def init_distributed_context(github_user: str):
    comm = MPI.COMM_WORLD
    rank = comm.Get_rank()
    size = comm.Get_size()
    if size != 4:
        raise RuntimeError(f"Expected exactly 4 MPI ranks (2x2 grid), got {size}.")

    # Bind process to GPU 0 on Google Colab
    cuda.select_device(0)
    gpu_uuid = get_gpu_uuid() if rank == 0 else None
    gpu_uuid = comm.bcast(gpu_uuid, root=0)
    instance_seed = derive_student_seed(github_user, gpu_uuid)

    cart = comm.Create_cart(dims=PROC_DIMS, periods=[False, False], reorder=False)
    cart_rank = cart.Get_rank()
    nbr_up, nbr_down = cart.Shift(0, 1)  # axis 0 = rows (y)
    nbr_left, nbr_right = cart.Shift(1, 1)  # axis 1 = columns (x)
    return SimpleNamespace(
        cart=cart,
        rank=cart_rank,
        coords=tuple(cart.Get_coords(cart_rank)),
        nbrs=(nbr_up, nbr_down, nbr_left, nbr_right),
        gpu_uuid=gpu_uuid,
        seed=instance_seed,
    )


# ==============================================================================
# TASK 2: Seeded Heterogeneous Subdomain Initialization (rank-local only)
# ==============================================================================
def initialize_subdomains(ctx):
    row_rank, col_rank = ctx.coords
    # Same seed on every rank -> every rank draws identical pulse parameters -> one consistent global field.
    rng = np.random.default_rng(ctx.seed)
    n_pulses = 3
    pulse_x = rng.uniform(0.2, 0.8, n_pulses)
    pulse_y = rng.uniform(0.2, 0.8, n_pulses)
    pulse_amp = rng.uniform(1.0, 2.0, n_pulses)
    pulse_sigma = rng.uniform(0.04, 0.08, n_pulses)
    phase_x, phase_y = rng.uniform(0.0, 2.0 * np.pi, 2)
    src_x, src_y = rng.uniform(0.3, 0.7, 2)

    # Local index l = 0..257 is global index rank_coord * 256 + l, so x = gj * dx covers [0, 1].
    y = (row_rank * NY_LOCAL + np.arange(NY_LOCAL + 2)) * DY
    x = (col_rank * NX_LOCAL + np.arange(NX_LOCAL + 2)) * DX
    X, Y = np.meshgrid(x, y)  # (258, 258): [i, j] -> (y_i, x_j)

    kappa = 0.16 + 0.08 * np.sin(2.0 * np.pi * X + phase_x) * np.cos(2.0 * np.pi * Y + phase_y)
    u = np.zeros_like(X)
    for cx, cy, amp, sig in zip(pulse_x, pulse_y, pulse_amp, pulse_sigma):
        u += amp * np.exp(-((X - cx) ** 2 + (Y - cy) ** 2) / (2.0 * sig * sig))
    src = 0.05 * np.exp(-((X - src_x) ** 2 + (Y - src_y) ** 2) / (2.0 * 0.03**2))

    nbr_up, nbr_down, nbr_left, nbr_right = ctx.nbrs
    if nbr_up == MPI.PROC_NULL:
        u[0, :] = 0.0
    if nbr_down == MPI.PROC_NULL:
        u[-1, :] = 0.0
    if nbr_left == MPI.PROC_NULL:
        u[:, 0] = 0.0
    if nbr_right == MPI.PROC_NULL:
        u[:, -1] = 0.0
    return u, kappa, src


# ==============================================================================
# TASK 3: GPU Halo Pack/Unpack Kernels & Pinned MPI Exchange
# ==============================================================================
@cuda.jit
def pack_halos_kernel(u, send, n):
    k = cuda.grid(1)
    if k < n:
        send[UP * n + k] = u[1, k + 1]  # first owned row
        send[DOWN * n + k] = u[n, k + 1]  # last owned row
        send[LEFT * n + k] = u[k + 1, 1]  # first owned column (stride 258 doubles)
        send[RIGHT * n + k] = u[k + 1, n]  # last owned column


@cuda.jit
def unpack_halos_kernel(u, recv, n):
    k = cuda.grid(1)
    if k < n:
        u[0, k + 1] = recv[UP * n + k]
        u[n + 1, k + 1] = recv[DOWN * n + k]
        u[k + 1, 0] = recv[LEFT * n + k]
        u[k + 1, n + 1] = recv[RIGHT * n + k]


def exchange_halos_async(st, u, stream):
    """Stage A + MPI half of Stage B: GPU pack -> async D2H into pinned memory -> post Irecv/Isend."""
    n = st.n
    pack_halos_kernel[st.pack_blocks, PACK_THREADS, stream](u, st.d_send, n)
    st.d_send.copy_to_host(st.h_send, stream=stream)
    stream.synchronize()  # MPI reads h_send from the host, so the D2H DMA must have landed
    reqs = [
        st.cart.Irecv([st.h_recv[d * n:(d + 1) * n], MPI.DOUBLE], source=st.nbrs[d], tag=OPPOSITE[d])
        for d in range(4)
    ]
    reqs += [
        st.cart.Isend([st.h_send[d * n:(d + 1) * n], MPI.DOUBLE], dest=st.nbrs[d], tag=d)
        for d in range(4)
    ]
    return reqs


def complete_halos(st, u, reqs, stream):
    """Halo half of Stage C: Waitall -> async H2D from pinned memory -> GPU unpack into ghosts 0 and n+1.

    A PROC_NULL receive never touches its zero-filled segment, so wall ghosts are re-written with the
    Dirichlet value u = 0 without any boundary branch.
    """
    MPI.Request.Waitall(reqs)
    st.d_recv.copy_to_device(st.h_recv, stream=stream)
    unpack_halos_kernel[st.pack_blocks, PACK_THREADS, stream](u, st.d_recv, st.n)


# ==============================================================================
# TASK 4: 2D Shared-Memory Tiled Stencil & Dual-Stream Orchestrator
# ==============================================================================
@cuda.jit
def tiled_stencil_kernel(u, u_new, kappa, src, r_start, r_end, c_start, c_end, dt, inv_dx2, inv_dy2, lam):
    tile = cuda.shared.array(shape=(SHARED_DIM, SHARED_DIM), dtype=float64)
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y
    i = r_start + cuda.blockIdx.y * TILE_DIM + ty
    j = c_start + cuda.blockIdx.x * TILE_DIM + tx
    sy = ty + 1
    sx = tx + 1

    # Stage only the box plus its 1-cell apron: the deep-interior launch then never reads ghost
    # cells, which stream_boundary may be unpacking at the same moment.
    row_ok = i <= r_end + 1
    col_ok = j <= c_end + 1
    if row_ok and col_ok:
        tile[sy, sx] = u[i, j]
        if ty == 0:
            tile[0, sx] = u[i - 1, j]
        if tx == 0:
            tile[sy, 0] = u[i, j - 1]
    if ty == TILE_DIM - 1 and i <= r_end and col_ok:
        tile[sy + 1, sx] = u[i + 1, j]
    if tx == TILE_DIM - 1 and j <= c_end and row_ok:
        tile[sy, sx + 1] = u[i, j + 1]
    cuda.syncthreads()

    if i <= r_end and j <= c_end:
        uc = tile[sy, sx]
        kc = kappa[i, j]
        div_y = (0.5 * (kappa[i + 1, j] + kc) * (tile[sy + 1, sx] - uc)
                 - 0.5 * (kappa[i - 1, j] + kc) * (uc - tile[sy - 1, sx])) * inv_dy2
        div_x = (0.5 * (kappa[i, j + 1] + kc) * (tile[sy, sx + 1] - uc)
                 - 0.5 * (kappa[i, j - 1] + kc) * (uc - tile[sy, sx - 1])) * inv_dx2
        u_new[i, j] = uc + dt * (div_x + div_y - lam * uc * uc * uc + src[i, j])


def launch_stencil(st, u, u_new, box, stream):
    r_start, r_end, c_start, c_end = box
    grid = ((c_end - c_start + TILE_DIM) // TILE_DIM, (r_end - r_start + TILE_DIM) // TILE_DIM)
    tiled_stencil_kernel[grid, (TILE_DIM, TILE_DIM), stream](
        u, u_new, st.kappa, st.src, r_start, r_end, c_start, c_end, st.dt, st.inv_dx2, st.inv_dy2, LAMBDA_DAMP
    )


def step_serialized(st, u, u_new):
    """Reference: full halo exchange first, then one stencil over all owned cells, single stream."""
    reqs = exchange_halos_async(st, u, st.stream_boundary)
    complete_halos(st, u, reqs, st.stream_boundary)
    launch_stencil(st, u, u_new, st.full_box, st.stream_boundary)
    st.stream_boundary.synchronize()


def step_overlapped(st, u, u_new):
    reqs = exchange_halos_async(st, u, st.stream_boundary)  # Stage A (+ posts Irecv/Isend)
    launch_stencil(st, u, u_new, st.interior_box, st.stream_interior)  # Stage B: needs no ghosts
    complete_halos(st, u, reqs, st.stream_boundary)  # Stage C
    for box in st.ring_boxes:
        launch_stencil(st, u, u_new, box, st.stream_boundary)
    st.stream_interior.synchronize()
    st.stream_boundary.synchronize()


# ==============================================================================
# TASK 5: Distributed Reductions, PTX Audit & Comparative Benchmark
# ==============================================================================
def global_diagnostics(cart, u, u_prev):
    """Global energy sum(u)*dx*dy and L2 residual ||u^n - u^(n-1)|| via Allreduce (no field gather)."""
    a = u.copy_to_host()[1:-1, 1:-1]
    b = u_prev.copy_to_host()[1:-1, 1:-1]
    local = np.array([a.sum() * DX * DY, ((a - b) ** 2).sum() * DX * DY])
    total = np.empty(2)
    cart.Allreduce(local, total, op=MPI.SUM)
    return float(total[0]), math.sqrt(total[1])


def run_mode(st, step_fn, steps):
    u = cuda.to_device(st.u0)
    u_new = cuda.to_device(st.u0)  # wall ghosts start at 0 in both buffers; the stencil never writes ghosts
    trace = [(0, *global_diagnostics(st.cart, u, u_new))]
    start, stop = cuda.event(), cuda.event()
    st.cart.Barrier()
    start.record(st.stream_boundary)
    for n in range(1, steps + 1):
        step_fn(st, u, u_new)
        u, u_new = u_new, u
        if n % REPORT_EVERY == 0:
            trace.append((n, *global_diagnostics(st.cart, u, u_new)))
    stop.record(st.stream_boundary)
    stop.synchronize()
    ms = st.cart.allreduce(cuda.event_elapsed_time(start, stop), op=MPI.MAX)
    return u.copy_to_host(), ms, trace


def ptx_audit():
    asm = tiled_stencil_kernel.inspect_asm()
    ptx = "\n".join(asm.values()) if isinstance(asm, dict) else str(asm)
    counts = {op: ptx.count(op) for op in ("st.shared.f64", "ld.shared.f64", "bar.sync")}
    return counts, all(c > 0 for c in counts.values())


def run_solver(github_user: str) -> bool:
    ctx = init_distributed_context(github_user)
    cart, rank = ctx.cart, ctx.rank

    if rank == 0:
        dev = cuda.get_current_device()
        print("=" * 78)
        print(f"GitHub user      : {github_user}")
        print(f"GPU UUID         : {ctx.gpu_uuid}")
        print(f"INSTANCE_SEED    : {ctx.seed} (0x{ctx.seed:016x})")
        print(f"GPU              : CC {dev.compute_capability}, {dev.MULTIPROCESSOR_COUNT} SMs, "
              f"{dev.ASYNC_ENGINE_COUNT} async copy engines")
        print(f"MPI ranks        : {cart.Get_size()} in a {PROC_DIMS[0]}x{PROC_DIMS[1]} Cartesian grid, "
              f"periods=[False, False], MPI.PROC_NULL={MPI.PROC_NULL}")
        print("=" * 78, flush=True)
    for r in range(cart.Get_size()):
        if r == rank:
            up, down, left, right = ctx.nbrs
            print(f"[rank {rank}] coords={ctx.coords} up={up} down={down} left={left} right={right} "
                  f"local=(258, 258) owned=(256, 256)", flush=True)
        cart.Barrier()

    u0, kappa, src = initialize_subdomains(ctx)
    kappa_max = cart.allreduce(float(kappa[1:-1, 1:-1].max()), op=MPI.MAX)
    kappa_min = cart.allreduce(float(kappa[1:-1, 1:-1].min()), op=MPI.MIN)
    dt_max = DX * DX / (4.0 * kappa_max)
    dt = CFL_SAFETY * dt_max

    n = NX_LOCAL
    st = SimpleNamespace(
        cart=cart,
        nbrs=ctx.nbrs,
        n=n,
        u0=u0,
        dt=dt,
        inv_dx2=1.0 / (DX * DX),
        inv_dy2=1.0 / (DY * DY),
        kappa=cuda.to_device(kappa),
        src=cuda.to_device(src),
        d_send=cuda.device_array(4 * n, dtype=np.float64),
        d_recv=cuda.device_array(4 * n, dtype=np.float64),
        h_send=cuda.pinned_array(4 * n, dtype=np.float64),
        h_recv=cuda.pinned_array(4 * n, dtype=np.float64),
        stream_boundary=cuda.stream(),
        stream_interior=cuda.stream(),
        pack_blocks=(n + PACK_THREADS - 1) // PACK_THREADS,
        full_box=(1, n, 1, n),
        interior_box=(2, n - 1, 2, n - 1),
        ring_boxes=((1, 1, 1, n), (n, n, 1, n), (2, n - 1, 1, 1), (2, n - 1, n, n)),
    )
    st.h_send[:] = 0.0
    st.h_recv[:] = 0.0  # pinned memory starts uninitialised; PROC_NULL segments must read as u = 0

    if rank == 0:
        print(f"dx = dy = 1/{GLOBAL_NX + 1} = {DX:.9e}")
        print(f"kappa range      : [{kappa_min:.6f}, {kappa_max:.6f}]")
        print(f"dt_max (CFL)     : dx^2 / (4 kappa_max) = {dt_max:.6e}")
        print(f"dt used          : {CFL_SAFETY} * dt_max = {dt:.6e}  ({STEPS} steps -> t = {STEPS * dt:.6e})",
              flush=True)

    run_mode(st, step_serialized, 1)  # JIT warm-up so compilation is not timed
    run_mode(st, step_overlapped, 1)

    ser_u, ser_ms, ser_trace = run_mode(st, step_serialized, STEPS)
    ovl_u, ovl_ms, ovl_trace = run_mode(st, step_overlapped, STEPS)
    parity = cart.allreduce(float(np.abs(ser_u - ovl_u).max()), op=MPI.MAX)
    ptx_counts, ptx_ok = ptx_audit()
    parity_ok = parity < 1e-12

    if rank == 0:
        for name, trace in (("serialized", ser_trace), ("overlapped", ovl_trace)):
            print(f"\nAllreduce diagnostics ({name}):")
            print(f"{'step':>6} {'energy E':>22} {'L2 residual':>22}")
            for s, e, r in trace:
                print(f"{s:>6} {e:>22.15e} {r:>22.15e}")
        print(f"\nserialized_ms    : {ser_ms:.3f}  ({ser_ms / STEPS:.4f} ms/step)")
        print(f"overlapped_ms    : {ovl_ms:.3f}  ({ovl_ms / STEPS:.4f} ms/step)")
        print(f"overlap speedup  : {ser_ms / ovl_ms:.3f}x")
        print(f"parity_max_diff  : {parity:.3e}  -> {'PASS' if parity_ok else 'FAIL'} (< 1e-12)")
        print(f"PTX audit        : {ptx_counts} -> {'PASS' if ptx_ok else 'FAIL'}")

        manifest = {
            "github_username": github_user,
            "gpu_uuid": ctx.gpu_uuid,
            "instance_seed": ctx.seed,
            "mpi_ranks": cart.Get_size(),
            "cart_dims": PROC_DIMS,
            "cart_periods": [False, False],
            "global_interior_shape": [GLOBAL_NY, GLOBAL_NX],
            "local_padded_shape": list(u0.shape),
            "tile_dim": TILE_DIM,
            "shared_tile_dim": SHARED_DIM,
            "steps": STEPS,
            "dx": DX,
            "dt": dt,
            "kappa_min": kappa_min,
            "kappa_max": kappa_max,
            "dt_max_cfl": dt_max,
            "lambda_damp": LAMBDA_DAMP,
            "energy_trace": [{"step": s, "energy": e, "l2_residual": r} for s, e, r in ovl_trace],
            "serialized_ms": ser_ms,
            "overlapped_ms": ovl_ms,
            "overlap_speedup": ser_ms / ovl_ms,
            "timing_method": "CUDA events on stream_boundary around the 400-step loop, max over ranks",
            "parity_max_diff": parity,
            "parity_passed": parity_ok,
            "ptx_counts": ptx_counts,
            "ptx_audit_passed": ptx_ok,
            "generated_utc": datetime.now(timezone.utc).isoformat(),
        }
        with open("submission_manifest.json", "w") as f:
            json.dump(manifest, f, indent=2)
        print("\nWrote submission_manifest.json", flush=True)
    return parity_ok and ptx_ok


if __name__ == "__main__":
    if len(sys.argv) != 2:
        raise SystemExit("usage: mpirun -np 4 python3 distributed_cuda_solver.py <github_username>")
    sys.exit(0 if run_solver(sys.argv[1]) else 1)


Writing distributed_cuda_solver.py


### Cell 4 — Multi-rank MPI + CUDA execution launcher

In [4]:
import subprocess

cmd = [
    "mpirun",
    "--allow-run-as-root",
    "--oversubscribe",
    "-np", "4",
    "python3",
    "distributed_cuda_solver.py",
    GITHUB_USERNAME,
]
proc = subprocess.run(cmd, capture_output=True, text=True)
print(proc.stdout)
if proc.returncode != 0:
    print("STDERR:\n", proc.stderr)
    raise RuntimeError(f"MPI execution failed with exit code {proc.returncode}")

GitHub user      : 230103278
GPU UUID         : GPU-9f464241-9201-a2c6-af69-f3df6f6794fd
INSTANCE_SEED    : 3157701663442476547 (0x2bd268eec4261e03)
GPU              : CC ComputeCapability(major=7, minor=5), 40 SMs, 3 async copy engines
MPI ranks        : 4 in a 2x2 Cartesian grid, periods=[False, False], MPI.PROC_NULL=-2
[rank 0] coords=(0, 0) up=-2 down=2 left=-2 right=1 local=(258, 258) owned=(256, 256)
[rank 1] coords=(0, 1) up=-2 down=3 left=0 right=-2 local=(258, 258) owned=(256, 256)
[rank 2] coords=(1, 0) up=0 down=-2 left=-2 right=3 local=(258, 258) owned=(256, 256)
[rank 3] coords=(1, 1) up=1 down=-2 left=2 right=-2 local=(258, 258) owned=(256, 256)
dx = dy = 1/513 = 1.949317739e-03
kappa range      : [0.080001, 0.240000]
dt_max (CFL)     : dx^2 / (4 kappa_max) = 3.958171e-06
dt used          : 0.8 * dt_max = 3.166537e-06  (400 steps -> t = 1.266615e-03)

Allreduce diagnostics (serialized):
  step               energy E            L2 residual
     0  9.651719477151445e-02  0.

### Generated manifest

In [5]:
!cat submission_manifest.json

{
  "github_username": "230103278",
  "gpu_uuid": "GPU-9f464241-9201-a2c6-af69-f3df6f6794fd",
  "instance_seed": 3157701663442476547,
  "mpi_ranks": 4,
  "cart_dims": [
    2,
    2
  ],
  "cart_periods": [
    false,
    false
  ],
  "global_interior_shape": [
    512,
    512
  ],
  "local_padded_shape": [
    258,
    258
  ],
  "tile_dim": 16,
  "shared_tile_dim": 18,
  "steps": 400,
  "dx": 0.001949317738791423,
  "dt": 3.1665369491083255e-06,
  "kappa_min": 0.08000051185745685,
  "kappa_max": 0.23999970364071793,
  "dt_max_cfl": 3.958171186385407e-06,
  "lambda_damp": 0.15,
  "energy_trace": [
    {
      "step": 0,
      "energy": 0.09651719477151445,
      "l2_residual": 0.0
    },
    {
      "step": 100,
      "energy": 0.09644360171981244,
      "l2_residual": 5.973907952516041e-05
    },
    {
      "step": 200,
      "energy": 0.09639302150884156,
      "l2_residual": 5.6381195827499535e-05
    },
    {
      "step": 300,
      "energy": 0.09634532372860705,
      "l2_resi

### Cell 5 — Forensic & analytical responses

### Analytical & Forensic Engineering Responses

> All measured values below come from this notebook's Colab run (Tesla T4 `GPU-9f464241-9201-a2c6-af69-f3df6f6794fd`, `INSTANCE_SEED` = 3157701663442476547) and its `submission_manifest.json`.

- **Question 1 (Memory Coalescing & Strided Column Packing):**
  The local field is a C-contiguous `(258, 258)` float64 array, so one row is 258 × 8 = **2,064 B** long. A halo **row** `u[1, 1:257]` is one contiguous 2,048 B run. A halo **column** `u[1:257, 1]` is 256 doubles spaced 2,064 B apart. The copy engine behind `cudaMemcpyAsync` moves one linear byte range per request, so a column cannot be described as a single DMA. It would need 256 separate 8-byte descriptors, a slow `cudaMemcpy2DAsync` with an 8-byte width, or a copy of the whole enclosing range.
  The same layout shows up at the warp level inside `pack_halos_kernel`. For a row, the 32 lanes of a warp read 32 consecutive doubles (256 B), which is 8 fully used 32-byte sectors. For a column, each lane's address is 2,064 B from its neighbour's, so the warp touches **32 different sectors** and uses 8 of every 32 bytes fetched (25 % sector efficiency). That cost is paid once, on the GPU, for only 256 elements. The write into the 1D send buffer is fully coalesced, so after packing all four strips are one contiguous 8,192 B buffer that leaves in a single D2H DMA and returns in a single H2D DMA.
  *Extra PCIe bytes of an unstrided bounding-box copy:* the smallest contiguous range containing `u[1:257, 1]` runs from flat index 1·258+1 to 256·258+1, which is 255·258 + 1 = **65,791 doubles = 526,328 B**. Only 2,048 B of that is useful, so **524,280 B (257×) are wasted per column**. In the 2 × 2 grid each rank has exactly one horizontal neighbour, so per step it sends one column and receives one: **1,048,560 extra bytes per rank per step**. Over 4 ranks × 400 steps that is **1,677,696,000 B ≈ 1.68 GB** of avoidable PCIe traffic. The packed scheme moves 2 × 8,192 B per rank per step, **26.2 MB** in total (all four segments are copied, PROC_NULL ones included). A bounding-box H2D into the ghost column would also be wrong, not just slow: it would overwrite 255 rows of owned cells with stale host data.

- **Question 2 (Shared-Memory Tile Indexing & Bank Conflict Analysis):**
  Thread `(tx, ty)` stages its centre at `smem[ty+1, tx+1]`. Lanes with `ty==0`, `ty==15`, `tx==0` or `tx==15` also store the apron cell on that side. The tile pitch is 18 × 8 = 144 B = 36 four-byte words, so element `smem[sy, sx]` starts at byte `8·(18·sy + sx)` and covers banks `b = 2·(18·sy+sx) mod 32` and `b+1`.

  | thread | role | `smem[sy, sx]` | byte offset | banks |
  |---|---|---|---|---|
  | (0,0) — warp 0, lane 0 | write centre | [1, 1] | 152 | 6, 7 |
  | | write top apron | [0, 1] | 8 | 2, 3 |
  | | write left apron | [1, 0] | 144 | 4, 5 |
  | | read N / S / E (W = left apron) | [0,1] / [2,1] / [1,2] / [1,0] | 8 / 296 / 160 / 144 | 2,3 / 10,11 / 8,9 / 4,5 |
  | (15,15) — warp 7, lane 31 | write centre | [16, 16] | 2,432 | 0, 1 |
  | | write bottom apron | [17, 16] | 2,576 | 4, 5 |
  | | write right apron | [16, 17] | 2,440 | 2, 3 |
  | | read N / S / W / E | [15,16] / [17,16] / [16,15] / [16,17] | 2,288 / 2,576 / 2,424 / 2,440 | 28,29 / 4,5 / 30,31 / 2,3 |

  On CC 7.5 there are 32 banks, each 4 B wide, and a 64-bit warp access is served as two 128-byte wavefronts, one per half-warp. Each warp of the 16 × 16 block covers two tile rows (`ty = 2w, 2w+1`), and each half-warp reads **16 consecutive doubles along a row**: 128 B covering all 32 banks exactly once. **Row access is therefore conflict-free.** The two wavefronts per warp are the minimum needed to move 256 B, not a 2-way conflict. **Column access does conflict 2-way.** Successive rows are 36 words apart, 36 mod 32 = 4, so element *k* of a column lands on bank 4k mod 32. Lanes *k* and *k+8* of a half-warp then collide (banks 0,4,…,28 repeat), giving 4 wavefronts per warp instead of 2. My kernel never sends a warp down a column. Vertical neighbours `smem[sy±1, sx]` are still read row-wise across the warp, and the column-apron stores come from only 2 lanes per warp (`tx==0` at `ty=2w, 2w+1`), whose banks differ by 4. So the stencil runs without bank conflicts. A 17-double pitch (34 words, stride 2 mod 32) would make columns conflict-free as well.

- **Question 3 (PTX Assembly Audit & Warp Synchronization):**
  `tiled_stencil_kernel.inspect_asm()` in this run emitted **5 × `st.shared.f64`**, **5 × `ld.shared.f64`** and **1 × `bar.sync`** (`ptx_audit_passed: true`). The 5 stores are the centre plus the four conditional apron stores (top, left, bottom, right). The 5 loads are the centre `uc` (loaded once and reused) plus the N/S/W/E neighbours. The single `bar.sync 0` is the one `cuda.syncthreads()`.
  Without that barrier there is a cross-warp **read-after-write hazard**. The 256 threads form 8 warps, and warp *w* stores tile rows `2w+1` and `2w+2`. Its stencil then reads rows `2w` and `2w+3`, which warps *w−1* and *w+1* store. Warp 0 also depends on the top apron (row 0) and warp 7 on the bottom apron (row 17), and every warp depends on the `tx==0/15` column-apron stores made by lanes in other half-warps. The SM's four schedulers issue warps independently, so warp *w* can reach its `ld.shared` before its neighbour has executed the matching `st.shared`. It then reads whatever the previous block left in that shared memory. The results become wrong and non-deterministic, which `parity_max_diff` and the energy trace would expose. Since Volta's independent thread scheduling, even dependencies between lanes of the same warp are not formally ordered without a barrier. `bar.sync` makes every shared store issued before it visible to every thread of the CTA after it.

- **Question 4 (Multi-Process Context Sharing on a Single Physical GPU):**
  `mpirun -np 4` starts four OS processes, and each calls `cuda.select_device(0)` and creates **its own CUDA context** on the same T4. Without MPS, kernels from different contexts never share the SMs at the same moment. The GPU **time-slices** between contexts, and only streams inside one context, such as my `stream_interior` and `stream_boundary`, can run kernels concurrently. The T4's **3** async copy engines (printed in the run header) and the PCIe link are shared by all four processes, so the ranks' D2H/H2D halo copies queue behind each other. On Colab the four ranks also share the VM's 2 vCPUs (`--oversubscribe`), and MPI busy-polls while it waits.
  Measured: `serialized_ms` = **917.9 ms** (**2.295 ms/step**) and `overlapped_ms` = **1286.1 ms** (**3.215 ms/step**), so `overlap_speedup` = **0.714×**. Overlap only pays when the work being hidden is longer than the extra cost of overlapping. Here the deep-interior update is 254² ≈ 64.5 k cells, a few µs of T4 time, while each halo message is 2 KB over shared memory, also a few µs. The overlapped step issues 9 GPU operations and 2 stream syncs per step (pack, D2H, interior, H2D, unpack, 4 ring kernels); the serialized step issues 5 operations and 1 sync. Each launch costs tens of µs of Python/Numba host overhead, and the four ring launches run 16 blocks with 15/16 of their threads idle. Context time-slicing also serialises one rank's interior kernel against the other ranks' work. On a single shared GPU the "overlapped" schedule therefore mostly adds overhead — in this run the overlapped schedule was **40 % slower** (+368 ms over 400 steps, +0.92 ms/step). With four dedicated GPUs over InfiniBand, each rank owns all its SMs and copy engines (no time-slicing), and production subdomains such as 4096² per GPU make interior compute take milliseconds against microseconds of halo transfer. Step time then falls from *compute + comm* towards *max(compute, comm)*, and with GPUDirect RDMA the host staging copies disappear as well.

- **Question 5 (Point-to-Point Message Deadlock & `MPI.PROC_NULL` Semantics):**
  A blocking `MPI_Send` may return as soon as the user buffer can be reused; it does not wait for the receive. Below the **eager limit** Open MPI copies the message into a pre-allocated receiver-side buffer and `Send` returns at once. The shared-memory BTL (`vader`/`sm`) defaults to 4 KiB; check with `ompi_info --param btl vader --level 9`. Above it the library switches to the **rendezvous** protocol. The sender sends only a ready-to-send header and **blocks until the matching receive has been posted**. Suppose every rank runs `Send(up); Send(down); Send(left); Send(right)` and only then the four `Recv`s. Rank 0 = (0,0) skips `Send(up)` (PROC_NULL) and blocks in `Send(down → rank 2)`, waiting for rank 2 to post `Recv(from up)`. Rank 2 is blocked in `Send(up → rank 0)`, waiting for rank 0's `Recv(from down)`. Neither rank ever reaches its `Recv`, which is a circular wait, and ranks 1↔3 deadlock the same way. A strip here is 256 × 8 = **2,048 B**, under the 4 KiB eager limit, so the unsafe version would probably pass at 512² by luck. Once a strip exceeds ~512 doubles (global grid ≥ 1024²), or messages cross the TCP BTL with its 64 KiB limit at larger sizes, it hangs. My code posts all `Irecv`s first, then all `Isend`s, and completes them with one `Waitall`, so no ordering between ranks can deadlock.
  For a send or receive whose peer is `MPI.PROC_NULL` (printed as `MPI.PROC_NULL=-2` by Open MPI 4.1.6 in this run), MPI completes the operation immediately, transfers nothing and leaves the receive buffer untouched. Rank (0,0) has `up = left = PROC_NULL`, `down = 2`, `right = 1`, yet it runs exactly the same 8-request loop as every other rank, with no `if` per edge and no special corner code. Because the pinned receive buffer is zero-filled at start-up, its UP and LEFT segments are never written. `unpack_halos_kernel` therefore copies 0.0 into ghost row 0 and ghost column 0 every step, which is precisely the homogeneous Dirichlet condition. The only explicit PROC_NULL tests in the program are in the one-time initialisation.

- **Question 6 (Explicit CFL Stability & Energy Conservation Proof):**
  Δx = Δy = 1/513 = 1.949318 × 10⁻³, so Δx² = 3.799840 × 10⁻⁶. With the run's κ_max = **0.2399997** (Allreduce MAX), Δt_max = Δx² / (4 κ_max) = **3.958171 × 10⁻⁶**. The solver uses Δt = 0.8 · Δt_max = **3.166537 × 10⁻⁶**, i.e. Δt / Δt_max = **0.8**, and the mesh Fourier number is r = κ_max Δt / Δx² = **0.2 ≤ 0.25**. In von Neumann terms the amplification factor of the 2D explicit diffusion operator is g = 1 − 4r[sin²(θx/2) + sin²(θy/2)]. Its worst case, the checkerboard mode, is g = 1 − 8r = **−0.6**, so |g| < 1 for every mode. Taking r > 0.25 would make |g| > 1 and the field would blow up in a growing checkerboard.
  Allreduce energy trace from this run:

  | step | E = Σ u ΔxΔy | L2 residual ‖uⁿ − uⁿ⁻¹‖ |
  |---|---|---|
  | 0 | 0.096517194772 | 0 |
  | 100 | 0.096443601720 | 5.973908e-05 |
  | 200 | 0.096393021509 | 5.638120e-05 |
  | 300 | 0.096345323729 | 5.344835e-05 |
  | 400 | 0.096298273295 | 5.084173e-05 |

  E decreases strictly from step to step (step-to-step drops of 7.36e-05, 5.06e-05, 4.77e-05, 4.71e-05; -0.227 % in total over 400 steps), and the residual also shrinks, which shows the scheme is dissipative and not oscillating. Physically, the conservative flux form makes interior diffusion move heat without creating it, so E changes only through three terms. Outflow through the u = 0 Dirichlet walls is small here, because the pulses are centred in [0.2, 0.8]². The cubic damping −λu³ with λ = 0.15 removes heat wherever u > 0, and with peak amplitudes of 1–2 it dominates. The weak source, 0.05·exp(−r²/2·0.03²), contributes only ∫S ≈ 2.8 × 10⁻⁴ per unit time. The trace is identical in serialized and overlapped mode, consistent with `parity_max_diff` = **0.0**.
